<a href="https://colab.research.google.com/github/nandkishor-vasi/NLP_assign/blob/main/Assign6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
!pip install scikit-learn

df = pd.read_csv(
    r'twitter_training.csv',
    header=None,
    names=['id', 'entity', 'sentiment', 'text']
)

print(df.head())
print(df.columns)

     id       entity  sentiment  \
0    id       entity  sentiment   
1  2401  Borderlands   Positive   
2  2401  Borderlands   Positive   
3  2401  Borderlands   Positive   
4  2401  Borderlands   Positive   

                                                text  
0                                               text  
1  im getting on borderlands and i will murder yo...  
2  I am coming to the borders and I will kill you...  
3  im getting on borderlands and i will kill you ...  
4  im coming on borderlands and i will murder you...  
Index(['id', 'entity', 'sentiment', 'text'], dtype='object')


In [4]:
df = df[['text', 'sentiment']]
df = df.dropna()

print(df.head())

                                                text  sentiment
0                                               text  sentiment
1  im getting on borderlands and i will murder yo...   Positive
2  I am coming to the borders and I will kill you...   Positive
3  im getting on borderlands and i will kill you ...   Positive
4  im coming on borderlands and i will murder you...   Positive


In [5]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['text'].apply(clean_text)

In [6]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['sentiment'] = le.fit_transform(df['sentiment'])

print(dict(zip(le.classes_, le.transform(le.classes_))))

{'Irrelevant': np.int64(0), 'Negative': np.int64(1), 'Neutral': np.int64(2), 'Positive': np.int64(3), 'sentiment': np.int64(4)}


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['sentiment'],
    test_size=0.2,
    random_state=42
)

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english', max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [9]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)

nb_pred = nb.predict(X_test_tfidf)
print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_pred))

Naive Bayes Accuracy: 0.6276351351351351


In [10]:
from sklearn.svm import LinearSVC

svm = LinearSVC()
svm.fit(X_train_tfidf, y_train)

svm_pred = svm.predict(X_test_tfidf)
print("SVM Accuracy:", accuracy_score(y_test, svm_pred))

SVM Accuracy: 0.6978378378378378


In [11]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=200)
lr.fit(X_train_tfidf, y_train)

lr_pred = lr.predict(X_test_tfidf)
print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_pred))

Logistic Regression Accuracy: 0.6771621621621622


In [12]:
def predict_sentiment(sentence):
    cleaned = clean_text(sentence)
    vec = tfidf.transform([cleaned])
    pred = lr.predict(vec)[0]
    return le.inverse_transform([pred])[0]

print(predict_sentiment("This game is absolutely amazing"))
print(predict_sentiment("I hate this update"))
print(predict_sentiment("check out this streamer"))
print(predict_sentiment("i enter that gunner seat"))

Positive
Negative
Irrelevant
Neutral
